<a href="https://colab.research.google.com/github/seekff/learn-python/blob/main/metaclass.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**metaclass 的真实含义：**

超越（Beyond）+ 变形（Change）

meta 不是“基本”的意思，而是“超越”和“改变”。

**metaclass 的能力：**

拦截类的创建 → 修改类本身 → 让类具备普通 class 无法拥有的能力。

**YAML 框架中的 metaclass 魔法**
目标：让任意 YAMLObject 子类自动支持序列化/反序列化
例如：


    class Monster(yaml.YAMLObject):
        yaml_tag = u'!Monster'
你无需手写 add_constructor(Monster)，YAML 会自动把 !Monster 映射到 Monster 类。

实现方式：YAMLObjectMetaclass 拦截类定义

在 metaclass 的 __init__ 中自动执行：


    cls.yaml_loader.add_constructor(cls.yaml_tag, cls.from_yaml)
效果：

所有 YAMLObject 子类在定义时都会被自动注册

使用者无需额外写注册代码

实现“超动态配置”，适用于大型项目的动态加载场景

In [3]:
import yaml

class Monster(yaml.YAMLObject):
  yaml_tag = u'!Monster'
  def __init__(self, name, hp, ac, attacks):
    self.name = name
    self.hp = hp
    self.attacks = attacks
    self.ac = ac
  def __repr__(self):
    return "%s(name=%r, hp=%r, ac=%r, attacks=%r)"%(
        self.__class__name__, self.name, self.hp, self.ac, self.attacks)

# yaml.load("""
# --- !Monster
# name: Cave spider
# hp: [2,6]    # 2d6
# ac: 16
# attacks: [BITE, HURT]
# """)

Monster(name='Cave spider', hp=[2,6], ac=16, attacks=['BITE','HURT'])

print(yaml.dump(
    Monster(name='Cave lizard', hp=[2,6], ac=16, attacks=['BITE','HURT'])
))


!Monster
ac: 16
attacks:
- BITE
- HURT
hp:
- 2
- 6
name: Cave lizard



**metaclass 的使用方式**

核心思想：

**通过自定义 metaclass，拦截类创建流程，在类生成时注入额外逻辑。**

典型结构：

    class MyMeta(type):
      def __init__(cls, name, bases, attrs):
        ...
当你写：


    class Foo(metaclass=MyMeta):
      ...
Python 实际执行的是：

    Foo = MyMeta(name, bases, attrs)


**Python 底层如何实现 metaclass？（三点）**

① 所有用户定义类都是 type 的实例

    type(MyClass)  # <class 'type'>

② 定义类 = 调用 type 的 call

    class MyClass: ...

# 等价于

    MyClass = type('MyClass', (), {...})

③ metaclass 是 type 的子类，替换了类的创建流程

当你指定：

    class Foo(metaclass=MyMeta):

Python 实际执行：

    Foo = MyMeta(name, bases, attrs)
    
因此 metaclass 可以：

修改类属性

自动注册类

改写继承关系

注入方法或行为

In [7]:
class MyClass:
  data = 1

instance = MyClass()

print(type(instance))

print(type(MyClass))

print(instance,MyClass)


MyClass = type('MyClass', (), {'data': 1})
instance = MyClass()
print(instance, MyClass)


print(instance.data)



<class '__main__.MyClass'>
<class 'type'>
<__main__.MyClass object at 0x7a461d7cab40> <class '__main__.MyClass'>
<__main__.MyClass object at 0x7a461d68b6e0> <class '__main__.MyClass'>
1



**Python Metaclass 工作流程图**

┌──────────────────────────────────────────────┐
│         
 **  1. 用户写下 class 语句  **            │
└──────────────────────────────────────────────┘
                │
                ▼
      Python 读取 class 代码块

      - 收集类体中的变量、方法

      - 形成 attrs 字典

      - 记录类名 name

      - 记录父类 bases
      
                │
                ▼
┌──────────────────────────────────────────────┐
│  
 **2. Python 决定使用哪个 metaclass **          │
└──────────────────────────────────────────────┘

优先级：

1. class 语句中显式指定 metaclass=...

2. 父类的 metaclass

3. 默认使用 type

                │
                ▼
┌──────────────────────────────────────────────┐
│  
** 3. 调用 metaclass.__call__  **               │
└──────────────────────────────────────────────┘

metaclass.__call__ 做三件事：

1. 调用 metaclass.__new__ 创建类对象（cls）

2. 调用 metaclass.__init__ 初始化类对象

3. 返回最终类对象

                │
                ▼
┌──────────────────────────────────────────────┐
│

 ** 4. metaclass.__new__(mcls, name, bases, attrs) │**
└──────────────────────────────────────────────┘

你可以在这里：

- 修改 attrs（添加/删除方法）

- 自动注册类

- 注入新行为

- 动态生成属性

                │
                ▼
┌──────────────────────────────────────────────┐
│   
**5. metaclass.__init__(cls, name, bases, attrs) │**
└──────────────────────────────────────────────┘

你可以在这里：

- 进一步修改类

- 执行副作用（如 YAML 自动注册）

- 记录类信息

                │
                ▼
┌──────────────────────────────────────────────┐
│   
6. 类对象创建完成，赋值给变量名            │
└──────────────────────────────────────────────┘

class Foo(metaclass=MyMeta):
    ...

最终效果：

Foo = MyMeta(name, bases, attrs)

                │
                ▼
┌──────────────────────────────────────────────┐
│   
7. 用户实例化类 → Foo()                    │
└──────────────────────────────────────────────┘

此时调用的是：

Foo.__call__ → Foo.__new__ → Foo.__init__

（注意：这是类的实例化，与 metaclass 无关）

